# Створення додатків для генерації зображень (Azure OpenAI)

Великі мовні моделі — це не лише генерація тексту. Ви також можете створювати зображення на основі текстових описів. Зображення як модальність корисні в медичних технологіях, архітектурі, туризмі, розробці ігор, маркетингу та багатьох інших сферах. У цьому уроці ми працюємо з сучасними моделями **GPT Image** на Microsoft Foundry.

## Цілі навчання

До кінця цього уроку ви зможете:

- Пояснити, що таке генерація зображень і де вона корисна.
- Зрозуміти сімейство моделей `gpt-image` та чим вони відрізняються від застарілих моделей DALL·E.
- Побудувати додаток для генерації зображень і редагування зображень.

## Що таке генерація зображень?

Моделі генерації зображень створюють зображення за текстовим запитом. Сучасні моделі, такі як `gpt-image`, під час навчання вивчають зв'язок між текстом і зображеннями, а потім ітеративно перетворюють випадковий шум у зображення, яке відповідає вашому запиту.

Дві відомі родини моделей для зображень:

- **`gpt-image` (OpenAI)** — сучасне покоління, яке використовується в цьому уроці. Підтримує генерацію з тексту у зображення та редагування зображень (інпейнтінг з маскою).
- **Midjourney** — популярна стороння модель зі своїм сервісом і робочим процесом у Discord.

> Старіші моделі OpenAI для зображень — **DALL·E 2** та **DALL·E 3** — застарілі. DALL·E 3 більше не доступна для нових розгортань, а функції, як-от `create_variation`, існували лише в DALL·E 2. Для нових застосунків використовуйте моделі `gpt-image`.

На Microsoft Foundry **`gpt-image-2`** є найновішою та найпотужнішою моделлю для зображень і рекомендованим варіантом за замовчуванням. Моделі `gpt-image-1.5` і `gpt-image-1-mini` також загальнодоступні.

> **Важливо:** Моделі `gpt-image` повертають створене зображення у форматі **base64** (`b64_json`), а не у вигляді URL. Ваш код декодує рядок base64 у байти і зберігає — URL для завантаження зображення відсутній.


## Створення вашого першого застосунку для генерації зображень

Отже, що потрібно для створення застосунку для генерації зображень? Вам потрібні наступні бібліотеки:

- **python-dotenv**, настійно рекомендується використовувати цю бібліотеку, щоб зберігати ваші секрети у файлі *.env* окремо від коду.
- **openai**, ця бібліотека потрібна для взаємодії з OpenAI API.
- **pillow**, для роботи з зображеннями у Python.

Якщо ви ще не зробили цього, дотримуйтесь інструкцій на сторінці [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst), щоб створити ресурс Microsoft Foundry та модель. Виберіть **gpt-image-2** як модель (найновіша модель для зображень Azure OpenAI; DALL·E є застарілою).

1. Створіть файл *.env* з таким вмістом:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Знайдіть цю інформацію на порталі Microsoft Foundry для вашого ресурсу у розділі "Deployments".



1. Зберіть вищезгадані бібліотеки у файл під назвою *requirements.txt* ось так:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Потім створіть віртуальне середовище та встановіть бібліотеки:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Для Windows використовуйте наступні команди, щоб створити й активувати ваше віртуальне середовище:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додайте наступний код у файл під назвою *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # імпортувати dotenv
    dotenv.load_dotenv()

    # налаштувати клієнта Azure OpenAI (Microsoft Foundry).
    # Для моделей зображень потрібна остання версія API — перевірте документацію Foundry для моделі, яку ви використовуєте.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Створити зображення за допомогою API генерації зображень
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введіть текст підказки тут
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # наприклад, "gpt-image-2"
        )
        # Встановити каталог для збереження зображення
        image_dir = os.path.join(os.curdir, 'images')

        # Якщо каталог не існує, створіть його
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Ініціалізувати шлях зображення (зверніть увагу, що тип файлу повинен бути png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # моделі gpt-image повертають зображення у форматі base64 (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Відобразити зображення у стандартному переглядачі зображень
        image = Image.open(image_path)
        image.show()

    # обробляти виключення
    except BadRequestError as err:
        print(err)

    ```

Давайте пояснимо цей код:

- Спочатку імпортуємо необхідні бібліотеки, включно з бібліотекою OpenAI, бібліотекою dotenv, модулем base64 і бібліотекою Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Далі завантажуємо змінні оточення із файлу *.env*.

    ```python
    # імпортувати dotenv
    dotenv.load_dotenv()
    ```

- Потім налаштовуємо клієнт Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Далі генеруємо зображення:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введіть ваш текст запиту тут
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Моделі `gpt-image` повертають зображення у вигляді **base64** рядка у `data[0].b64_json`. Ми декодуємо його в байти та записуємо у файл — посилання для завантаження відсутнє.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Нарешті, відкриваємо зображення і використовуємо стандартний переглядач зображень, щоб його відобразити:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Більше деталей про генерацію зображення

Розглянемо параметри `images.generate`:

- **prompt**, це текстовий запит, який використовується для генерації зображення. Тут "Кролик на коні, тримає льодяник, на туманній галявині, де ростуть нарциси".
- **size**, це розмір згенерованого зображення (наприклад `1024x1024`, `1536x1024`, `1024x1536` або `"auto"`).
- **n**, кількість згенерованих зображень. Тут ми генеруємо одне.
- **model**, це назва вашого розгортання моделі зображень (наприклад `gpt-image-2`).

> Моделі зображень не приймають параметр `temperature` — це керування генерацією тексту. Щоб отримати різноманітність, викликайте API знову; щоб зменшити різноманітність, зробіть запит більш конкретним.

## Додаткові можливості генерації зображень

Ви побачили, як згенерувати зображення кількома рядками Python. Моделі `gpt-image` також можуть **редагувати** існуюче зображення. Завдяки наданню зображення, опціональної **маски** (яка вказує область для змін) та запиту, ви можете змінити частину зображення — наприклад, додати капелюх нашому кролику.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# редагування також повертаються у форматі base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Початкове зображення містить лише кролика; фінальне зображення має капелюх на кролику.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
